# 00 Setup data
Downloads Spider, installs NLTK data, verifies the dataset, and runs a gold-vs-gold smoke check of the official evaluator.

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from common import config, env, data
from common.config import OFFICIAL_DIR, DATA_DIR, resolve_limit

# Run_Config
RUN_NAME = "00_setup_data"
LIMIT = resolve_limit(None)
SPIDER_GDRIVE_ID = "1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J"  # from taoyds/spider README
print("ROOT", ROOT, "DATA_DIR", DATA_DIR, "OFFICIAL_DIR", OFFICIAL_DIR, "LIMIT", LIMIT)

ROOT /home/nguyen/School/CISC_451/assignment1 DATA_DIR /home/nguyen/School/CISC_451/assignment1/data/spider OFFICIAL_DIR /home/nguyen/School/CISC_451/test-suite-sql-eval LIMIT 10


## Download Spider (idempotent)

In [2]:
import shutil, zipfile

def data_present():
    return all((DATA_DIR / p).exists() for p in ("dev.json", "tables.json", "dev_gold.sql", "database"))

with env.track_run(RUN_NAME + ":download", LIMIT):
    if data_present():
        print("Spider data already present; skipping download.")
    else:
        import gdown
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        zpath = DATA_DIR.parent / "spider_data.zip"
        if not zpath.exists():
            gdown.download(id=SPIDER_GDRIVE_ID, output=str(zpath), quiet=False)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(DATA_DIR.parent / "_extract")
        ex = DATA_DIR.parent / "_extract"
        src = next(p.parent for p in ex.rglob("dev.json") if "__MACOSX" not in p.parts)
        for item in src.iterdir():
            dst = DATA_DIR / item.name
            if not dst.exists():
                shutil.move(str(item), str(dst))
        shutil.rmtree(ex)
        zpath.unlink()
        assert data_present(), "Extraction did not produce expected files"
    print(sorted(p.name for p in DATA_DIR.iterdir()))

Spider data already present; skipping download.
['.DS_Store', 'README.txt', 'database', 'dev.json', 'dev_gold.sql', 'tables.json', 'test.json', 'test_database', 'test_gold.sql', 'test_tables.json', 'train_gold.sql', 'train_others.json', 'train_spider.json']


If `gdown` fails, download `spider_data.zip` manually from the Spider site/Google Drive link in the taoyds/spider README and extract so that `data/spider/dev.json`, `tables.json`, `dev_gold.sql`, and `database/` exist.

In [3]:
import nltk
for pkg in ("punkt", "punkt_tab"):
    nltk.download(pkg, quiet=True)
print("nltk ok")

nltk ok


## Verify dataset

In [4]:
summary = data.verify_dataset()
print(summary["n_examples"], summary["n_gold"], summary["n_databases"])
(OFFICIAL_DIR / "tmp").mkdir(exist_ok=True)

1034 1034 20


## Gold-vs-gold Official_CLI smoke check

In [5]:
import subprocess, re, tempfile
with env.track_run(RUN_NAME + ":gold_vs_gold", LIMIT):
    tmpd = Path(tempfile.mkdtemp())
    gold = data.write_gold(tmpd / "gold.txt", LIMIT)
    pred = tmpd / "pred.txt"
    pred.write_text("".join(s + "\n" for s, _ in data.gold_lines(LIMIT)), encoding="utf-8")
    r = subprocess.run([sys.executable, "evaluation.py", "--gold", str(gold), "--pred", str(pred),
                        "--db", str(DATA_DIR / "database"), "--table", str(DATA_DIR / "tables.json"),
                        "--etype", "all"], cwd=OFFICIAL_DIR, capture_output=True, text=True)
    out = r.stdout
    print(out[-3000:]); print(r.stderr[-2000:])
    def overall(label):
        line = next(l for l in out.splitlines() if l.strip().startswith(label))
        return float(line.split()[-1])
    ex, em = overall("execution"), overall("exact match")
    print("EX", ex, "EM", em)
    assert ex == 1.0 and em == 1.0

    1.000                0.000                0.000                1.000               
IUEN                 0.000                0.000                0.000                0.000                0.000               
keywords             1.000                1.000                0.000                0.000                1.000               
---------------------- PARTIAL MATCHING RECALL ----------------------
select               1.000                1.000                0.000                0.000                1.000               
select(no AGG)       1.000                1.000                0.000                0.000                1.000               
where                1.000                1.000                0.000                0.000                1.000               
where(no OP)         1.000                1.000                0.000                0.000                1.000               
group(no Having)     0.000                0.000                0.000                0.

## Wrapper-vs-CLI validation (Req 11.1–11.3)
Deterministic mix of 50 predictions on the first 50 dev examples: gold, perturbed (LIMIT 1 / wrong projection), unparsable text, erroring SQL, and Placeholder_SQL. The wrapper totals must match the official CLI.

In [6]:
import re
import pandas as pd
from common import evaluate
from common.extract import PLACEHOLDER_SQL

N_VAL = 50
val_gold = data.gold_lines(N_VAL)

def make_pred(i, g):
    kind = ["gold", "limit", "unparsable", "error", "placeholder",
            "gold", "wrong_column", "unparsable", "error", "gold"][i % 10]
    if kind == "gold":
        return kind, g
    if kind == "limit":
        return kind, (re.sub(r"(?i)\bLIMIT\s+\d+", "LIMIT 2", g) if re.search(r"(?i)\bLIMIT\b", g)
                      else g + " LIMIT 1")
    if kind == "wrong_column":
        return kind, re.sub(r"(?is)^SELECT\s+.*?\s+FROM\s", "SELECT * FROM ", g, count=1)
    if kind == "unparsable":
        return kind, "I cannot answer this question"
    if kind == "error":
        return kind, "SELECT no_such_column FROM no_such_table"
    return kind, PLACEHOLDER_SQL

kinds, val_preds = zip(*(make_pred(i, g) for i, (g, _) in enumerate(val_gold)))
val_dir = config.ROOT_DIR / "outputs" / "_wrapper_cli_validation"
val_dir.mkdir(parents=True, exist_ok=True)
val_gold_path = data.write_gold(val_dir / "gold.txt", N_VAL)
val_pred_path = val_dir / "preds.txt"
val_pred_path.write_text("".join(p + "\n" for p in val_preds), encoding="utf-8")
print(pd.Series(kinds).value_counts().to_dict())

{'gold': 15, 'unparsable': 10, 'error': 10, 'limit': 5, 'placeholder': 5, 'wrong_column': 5}


In [7]:
with env.track_run(RUN_NAME + ":wrapper_vs_cli", LIMIT):
    ext = [{"extraction_status": "missing" if k == "placeholder" else "ok", "kind": k} for k in kinds]
    res = evaluate.evaluate_predictions(list(val_preds), val_gold, ext, None, val_dir,
                                        run_name="wrapper_cli_validation")
    cli = evaluate.run_official_cli(val_gold_path, val_pred_path)
    mismatches = evaluate.compare_totals(res.table, cli)
    evaluate.record_cli_check(val_dir, mismatches)

val_table = pd.DataFrame({
    "level": evaluate.LEVELS,
    "count_wrapper": [int(res.table.loc[l, "count"]) for l in evaluate.LEVELS],
    "count_cli": [int(cli["count"][l]) for l in evaluate.LEVELS],
    "ex_wrapper": [round(float(res.table.loc[l, "ex"]), 4) for l in evaluate.LEVELS],
    "ex_cli": [cli["execution"][l] for l in evaluate.LEVELS],
    "em_wrapper": [round(float(res.table.loc[l, "em"]), 4) for l in evaluate.LEVELS],
    "em_cli": [cli["exact match"][l] for l in evaluate.LEVELS],
})
res_dir = config.results_root(None)
res_dir.mkdir(parents=True, exist_ok=True)
val_table.to_csv(res_dir / "wrapper_cli_validation.csv", index=False)
(res_dir / "wrapper_cli_validation.md").write_text("\n".join(["| " + " | ".join(val_table.columns) + " |", "|" + "---|" * len(val_table.columns)]
    + ["| " + " | ".join(str(v) for v in r) + " |" for r in val_table.itertuples(index=False)]) + "\n", encoding="utf-8")
print(val_table.to_string(index=False))
print("parse errors:", res.summary["n_parse_error"], "invalid:", res.summary["n_invalid"])
assert mismatches == [], mismatches
print("Wrapper totals match the official CLI.")

 level  count_wrapper  count_cli  ex_wrapper  ex_cli  em_wrapper  em_cli
  easy              6          6      0.6667   0.667      0.5000   0.500
medium             27         27      0.3704   0.370      0.3333   0.333
  hard             13         13      0.1538   0.154      0.1538   0.154
 extra              4          4      0.5000   0.500      0.2500   0.250
   all             50         50      0.3600   0.360      0.3000   0.300
parse errors: 25 invalid: 25
Wrapper totals match the official CLI.


## Environment snapshot

In [8]:
import json
print(json.dumps(env.snapshot(), indent=2))

{
  "machine": {
    "os": "Linux",
    "kernel": "6.18.40.1-microsoft-standard-WSL2",
    "is_wsl": true,
    "distro": "Ubuntu 24.04.1 LTS",
    "cpu_model": "13th Gen Intel(R) Core(TM) i7-13620H",
    "logical_cpus": 16,
    "ram_gib": 7.61,
    "ram_note": "RAM allocated to the WSL VM (from /proc/meminfo), not total host RAM",
    "python": "3.12.3",
    "python_implementation": "CPython"
  },
  "packages": {
    "openai": "3.26.0",
    "pandas": "3.0.6",
    "scipy": "1.18.1",
    "statsmodels": "0.15.0",
    "nltk": "3.10.3",
    "sqlparse": "0.6.0"
  },
  "official_commit": "e97acc546ecbee8fa27fa8dbf025ef61493a876c",
  "inference": {
    "access": "api",
    "service": "OpenRouter",
    "base_url": "https://openrouter.ai/api/v1",
    "remote_hardware_controlled": false,
    "local_role": "orchestration, SQL execution, evaluation"
  }
}
